In [0]:
%pip install geopandas shapely

In [0]:
%restart_python

In [0]:
import requests
import json
import geopandas as gpd
import pandas as pd

from shapely import wkt
from datetime import datetime, timezone

WGS84_CRS = "EPSG:4326"
PROJECTED_CRS = "EPSG:3310"

SILVER_STATIC_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "silver/zone_static_features"
)

BRONZE_WEATHER_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "bronze/weather"
)

SILVER_WEATHER_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "silver/current_weather"
)

WEATHER_API_URL = (
    "https://api.open-meteo.com/v1/forecast"
)

BATCH_SIZE = 50

print("Weather configuration loaded.")

In [0]:
static_pdf = (
    spark.read
    .format("delta")
    .load(SILVER_STATIC_PATH)
    .toPandas()
)

static_pdf["geometry"] = (
    static_pdf["geometry_wkt"]
    .apply(wkt.loads)
)

zones = gpd.GeoDataFrame(
    static_pdf.drop(columns=["geometry_wkt"]),
    geometry="geometry",
    crs=PROJECTED_CRS
)

print("Zones loaded:", len(zones))
print("Unique zones:", zones["zone_id"].nunique())

In [0]:
zone_weather_points = zones[
    ["zone_id", "geometry"]
].copy()

zone_weather_points["geometry"] = (
    zone_weather_points
    .geometry
    .representative_point()
)

zone_weather_points = (
    zone_weather_points
    .to_crs(WGS84_CRS)
)

zone_weather_points["longitude"] = (
    zone_weather_points.geometry.x
)

zone_weather_points["latitude"] = (
    zone_weather_points.geometry.y
)

print("Weather points:", len(zone_weather_points))

print(
    zone_weather_points[
        [
            "zone_id",
            "latitude",
            "longitude"
        ]
    ].head()
)

In [0]:
weather_rows = []
raw_batches = []

pipeline_run_timestamp = (
    datetime.now(timezone.utc)
    .isoformat()
)

for start in range(
    0,
    len(zone_weather_points),
    BATCH_SIZE
):

    batch = zone_weather_points.iloc[
        start:start + BATCH_SIZE
    ].copy()

    batch_number = (
        start // BATCH_SIZE
    ) + 1

    params = {
        "latitude": ",".join(
            batch["latitude"]
            .astype(str)
        ),
        "longitude": ",".join(
            batch["longitude"]
            .astype(str)
        ),
        "current": (
            "temperature_2m,"
            "relative_humidity_2m,"
            "wind_speed_10m,"
            "wind_gusts_10m"
        ),
        "temperature_unit": "fahrenheit",
        "wind_speed_unit": "mph",
        "timezone": "America/Los_Angeles"
    }

    response = requests.get(
        WEATHER_API_URL,
        params=params,
        timeout=60
    )

    response.raise_for_status()

    results = response.json()

    if isinstance(results, dict):
        results = [results]

    if len(results) != len(batch):
        raise ValueError(
            f"Batch {batch_number}: "
            "weather response count does not match zone count."
        )

    # Preserve raw API response for Bronze
    raw_batches.append({
        "batch_number": batch_number,
        "pipeline_run_timestamp": pipeline_run_timestamp,
        "zone_count": len(batch),
        "raw_json": json.dumps(results)
    })

    for i, result in enumerate(results):

        current = result["current"]
        zone = batch.iloc[i]

        weather_rows.append({
            "zone_id": zone["zone_id"],
            "latitude": float(zone["latitude"]),
            "longitude": float(zone["longitude"]),
            "weather_timestamp": current["time"],
            "temperature_f": current["temperature_2m"],
            "relative_humidity_pct":
                current["relative_humidity_2m"],
            "wind_speed_mph":
                current["wind_speed_10m"],
            "wind_gust_mph":
                current["wind_gusts_10m"],
            "weather_timezone":
                "America/Los_Angeles",
            "pipeline_run_timestamp":
                pipeline_run_timestamp
        })

    print(
        f"Batch {batch_number} complete: "
        f"{len(batch)} zones"
    )

print("Weather API retrieval complete.")

In [0]:
weather = pd.DataFrame(
    weather_rows
)

print(
    "Weather rows:",
    len(weather)
)

print(
    "Unique zones:",
    weather["zone_id"].nunique()
)

display(weather.head())

In [0]:
weather_columns = [
    "weather_timestamp",
    "temperature_f",
    "relative_humidity_pct",
    "wind_speed_mph",
    "wind_gust_mph"
]

print("Missing weather values:")
print(
    weather[
        weather_columns
    ].isna().sum()
)

print(
    "\nInvalid humidity:",
    (
        ~weather[
            "relative_humidity_pct"
        ].between(0, 100)
    ).sum()
)

print(
    "Negative wind speed:",
    (
        weather[
            "wind_speed_mph"
        ] < 0
    ).sum()
)

print(
    "Negative gust speed:",
    (
        weather[
            "wind_gust_mph"
        ] < 0
    ).sum()
)

print(
    "Duplicate zones:",
    weather["zone_id"]
    .duplicated()
    .sum()
)

In [0]:
bronze_weather_df = spark.createDataFrame(
    pd.DataFrame(raw_batches)
)

run_id = (
    datetime.now(timezone.utc)
    .strftime("%Y%m%d_%H%M%S")
)

bronze_output_path = (
    f"{BRONZE_WEATHER_PATH}/"
    f"load_timestamp={run_id}"
)

(
    bronze_weather_df.write
    .format("delta")
    .mode("overwrite")
    .save(bronze_output_path)
)

print(
    "Bronze weather written:",
    bronze_output_path
)

In [0]:
weather_spark_df = spark.createDataFrame(
    weather
)

(
    weather_spark_df.write
    .format("delta")
    .mode("overwrite")
    .save(SILVER_WEATHER_PATH)
)

print("Silver current weather written successfully")
print("Path:", SILVER_WEATHER_PATH)

In [0]:
check_weather = (
    spark.read
    .format("delta")
    .load(SILVER_WEATHER_PATH)
)

print(
    "Rows:",
    check_weather.count()
)

print(
    "Unique zones:",
    check_weather
    .select("zone_id")
    .distinct()
    .count()
)

display(
    check_weather
    .select(
        "zone_id",
        "weather_timestamp",
        "temperature_f",
        "relative_humidity_pct",
        "wind_speed_mph",
        "wind_gust_mph"
    )
    .limit(10)
)

In [0]:
print("Weather rows:", len(weather))
print("Unique zones:", weather["zone_id"].nunique())

print("\nMissing values:")
print(
    weather[
        [
            "weather_timestamp",
            "temperature_f",
            "relative_humidity_pct",
            "wind_speed_mph",
            "wind_gust_mph"
        ]
    ].isna().sum()
)

print(
    "\nInvalid humidity:",
    (~weather["relative_humidity_pct"].between(0, 100)).sum()
)

print(
    "Negative wind speed:",
    (weather["wind_speed_mph"] < 0).sum()
)

print(
    "Negative gust speed:",
    (weather["wind_gust_mph"] < 0).sum()
)

print(
    "Duplicate zones:",
    weather["zone_id"].duplicated().sum()
)

In [0]:
SILVER_WEATHER_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "silver/current_weather"
)

silver_weather_check = (
    spark.read
    .format("delta")
    .load(SILVER_WEATHER_PATH)
)

silver_weather_check.select(
    "pipeline_run_timestamp"
).distinct().show(truncate=False)